# iNat ↔ Wikidata taxon matching — report

Results, negative-sampling story, and error analysis. See `docs/inat-wikidata-match-spec.md`.

This notebook is populated incrementally, one section per milestone (spec §7), as each lands —
not written retroactively at the end.

## Milestone 1 — Ingest

Read `~/.cache/wikidata-inat-checker/taxa.db` (built by the sibling
[wikidata-inat-checker](https://github.com/Livia-Rasp/wikidata-inat-checker) repo) read-only,
and build a cached normalised-name + FTS5 trigram lookup table from it. See `src/normalize.py`
and `src/candidates.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.candidates import (
    build_lookup_cache,
    lookup_by_normalized_name,
    DEFAULT_TAXA_DB_PATH,
    DEFAULT_CACHE_PATH,
)
from src.normalize import normalize_name

conn = build_lookup_cache()
print(f"taxa index:   {DEFAULT_TAXA_DB_PATH}")
print(f"lookup cache: {DEFAULT_CACHE_PATH}")

taxa index:   /home/livia/.cache/wikidata-inat-checker/taxa.db
lookup cache: /home/livia/repos/xgboost-inat-wikidata-match/data/lookup.sqlite


### Corpus size and rank distribution

Confirms the index matches what the ingest exploration found: 1.4M active taxa, `rank_level`
and `active` not persisted (dropped by the Node builder — flagged as a gap to close in
milestone 4, features).

In [2]:
total = conn.execute("SELECT COUNT(*) FROM taxa_normalized").fetchone()[0]
print(f"{total:,} taxa in the index\n")
print(f"{'rank':15s}count")
for rank, n in conn.execute(
    "SELECT rank, COUNT(*) c FROM taxa_normalized GROUP BY rank ORDER BY c DESC LIMIT 10"
):
    print(f"{rank:15s}{n:,}")

1,418,443 taxa in the index

rank           count
species        1,116,164
genus          143,042
subspecies     87,842
variety        18,859
family         10,719
subgenus       6,896
hybrid         6,518
tribe          6,070
subfamily      5,472
section        4,842


### The hemihomonym check (milestone 1's acceptance test)

`Prunella` names both a bird genus (accentors, Prunellidae) and a mint genus (self-heal,
Lamiaceae). Name-only matching can't tell them apart — only ancestry can. This is exactly the
case candidate strategy 1 (spec §2) must surface both candidates for, rather than stopping at
the first hit.

In [3]:
matches = lookup_by_normalized_name(conn, "prunella")
print(f"{len(matches)} match(es) for 'prunella':")
for row in matches:
    print(f"  {row}")

2 match(es) for 'prunella':
  {'taxon_id': '13982', 'name': 'Prunella', 'rank': 'genus', 'ancestry': '48460/1/2/355675/3/7251/71358'}
  {'taxon_id': '52765', 'name': 'Prunella', 'rank': 'genus', 'ancestry': '48460/47126/211194/47125/47124/48151/48623/520502/918917/919181'}


### Name normalisation examples (spec §1)

Authorship stripping, infraspecific connectors, hybrid markers, and the gender-stripped epithet
stem — the last of these is what makes `Acer rubrum` / `Acer ruber` collapse to the same stem
despite disagreeing on gender.

In [4]:
cases = [
    "Rosa canina L.",
    "Acer rubrum",
    "Acer ruber",
    "Rosa canina subsp. dumetorum",
    "Salix x sepulcralis",
    "\u00d7Fragaria",
    "Bellis perennis (L.) DC.",
]
for c in cases:
    n = normalize_name(c)
    print(f"{c!r:35} -> {n.normalized!r:35} stem={n.epithet_stem!r:12} hybrid={n.hybrid}")

'Rosa canina L.'                    -> 'rosa canina'                       stem='canin'      hybrid=False
'Acer rubrum'                       -> 'acer rubrum'                       stem='rubr'       hybrid=False
'Acer ruber'                        -> 'acer ruber'                        stem='rubr'       hybrid=False
'Rosa canina subsp. dumetorum'      -> 'rosa canina subsp dumetorum'       stem='canin'      hybrid=False
'Salix x sepulcralis'               -> 'salix sepulcralis'                 stem='sepulcral'  hybrid=True
'×Fragaria'                         -> 'fragaria'                          stem=None         hybrid=True
'Bellis perennis (L.) DC.'          -> 'bellis perennis'                   stem='perenn'     hybrid=False


### Observation: how much genuine name-collision exists, and one normalisation gap it surfaced

48,280 normalised names (3.4% of the index) are shared by 2+ taxa — the raw material for
hemihomonym-style ambiguity that candidate generation and the taxonomic-agreement features
(spec §4) need to resolve.

But the single most-duplicated name, `cortinarius` (501 rows), isn't really 501 genuine
homonyms — it's a normalisation gap. Provisional/unresolved iNat species like `Cortinarius sp.
'AZ19'` fail the authorship-boundary heuristic in `normalize.py` at `sp.` and get truncated back
to just the genus, `cortinarius`, colliding with the real genus-rank entry and each other. This
doesn't affect the milestone 1 acceptance check, but it inflates the homonym-group count above
and is worth a real decision before milestone 3 (candidate generation): either teach
`normalize.py` to recognise and drop provisional-name taxa specifically, or filter `sp.`/quoted
provisional epithets out of candidate generation entirely, since they have no stable Wikidata
counterpart to match against anyway.

In [5]:
homonym_groups = conn.execute(
    """
    SELECT COUNT(*) FROM (
        SELECT normalized_name FROM taxa_normalized GROUP BY normalized_name HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]
print(f"{homonym_groups:,} normalized names shared by 2+ taxa ({homonym_groups/total:.1%} of the index)")

dup = conn.execute(
    """
    SELECT normalized_name, COUNT(*) c FROM taxa_normalized
    GROUP BY normalized_name HAVING c > 1 ORDER BY c DESC LIMIT 5
    """
).fetchall()
print("most-duplicated normalized names:", dup)

sample = conn.execute(
    "SELECT taxon_id, name, rank FROM taxa_normalized WHERE normalized_name='cortinarius' AND rank='species' LIMIT 3"
).fetchall()
print("sample of what's actually behind 'cortinarius':", sample)

48,280 normalized names shared by 2+ taxa (3.4% of the index)
most-duplicated normalized names: [('cortinarius', 501), ('hygrocybe', 216), ('ramaria', 176), ('inocybe', 167), ('amanita', 161)]
sample of what's actually behind 'cortinarius': [('1666456', "Cortinarius sp. 'AZ19'", 'species'), ('1668233', "Cortinarius sp. 'PNW101'", 'species'), ('1662842', "Cortinarius sp. 'IN10'", 'species')]
